# 핵심 주장 검증 — 패혈증 실데이터로 따라가기

PhysioNet Challenge 2019 (패혈증 조기예측, 환자 20,336명) 으로 이 프로젝트의
핵심 주장을 직접 검증합니다.

> **핵심 주장: "NEWS만큼 잡아내면서, 헛알람은 더 적다."**
>
> 정확도를 높이겠다는 게 아닙니다. 같은 만큼 잡되 덜 시끄럽게 하겠다는 것입니다.

## 04번 노트북과 무엇이 다른가

`04_challenge2012_timeseries.ipynb`는 파이프라인의 *구조*를 배우는 노트북이었습니다.
이 노트북은 그 파이프라인으로 *주장을 검증*합니다. 그래서 04번에 없던 절이 셋 있습니다.

| | 04번 (Challenge 2012) | 이 노트북 (Challenge 2019) |
|---|---|---|
| 사건 | 원내 사망 | 패혈증 발병 |
| 사건 시각 기록 | **일 단위** (거침) | **시간 단위** (정확) |
| 규모 | 4,000명 | 20,336명 |
| 목적 | 파이프라인 이해 | **주장 검증** |
| 새로 배울 것 | — | 알람 부담의 함정 · seed 재현성 · 라벨 해상도 |

## 왜 2012이 아니라 2019인가

04번 데이터는 사망 시각을 "입원 3일째" 처럼 **날짜까지만** 기록합니다. 그런데 우리는
"6시간 안에 위험해질 사람"을 가르치고 있었습니다.

> **"화요일에 돌아가셨다"는 것만 알면서 "몇 시에 위험해지는지 맞혀봐"라고 가르친 셈**입니다.
> 정답지가 하루 단위로 뭉개져 있으니 모델이 배울 게 없습니다.
> 실제로 그 데이터에서는 학습하지 않는 NEWS 점수표한테 졌습니다.

Challenge 2019는 발병 시각이 시간 단위라 이 문제가 없습니다. **주장을 검증할 수 있는
데이터는 이쪽입니다.**

## 오늘 할 것

| 절 | 내용 | 왜 중요한가 |
|:--:|---|---|
| 1–2 | 데이터 로드와 EDA | 발병 전에 활력징후가 실제로 변하는지 |
| 3–4 | 윈도우 · 환자 단위 분할 | 04번 복습 (빠르게) |
| 5 | XGBoost vs NEWS | 이기는가 |
| 6 | **알람 부담과 그 함정** | **핵심 주장. 여기서 함정을 발견합니다** |
| 7 | **seed 재현성** | **한 번 이긴 게 운인지 실력인지** |
| 8 | lead-time | 개입할 시간이 있는가 |
| 9 | SHAP | 왜 위험한가 |

> 사건은 **패혈증 발병**이지 심정지가 아닙니다. 여기 숫자는 "방법이 실데이터에서
> 작동한다"는 근거이지, 경북대 심정지 성능이 아닙니다.

> 플롯 제목은 영어입니다 (서버에 한글 폰트가 없으면 □로 깨짐). 설명은 마크다운에 한글로.

## 0. 설정

`DATA_DIR`를 서버의 실제 경로로 맞추세요. 데이터가 없으면 다음 셀이 받는 방법을 알려줍니다.

`MAX_FILES`가 이 노트북의 속도를 좌우합니다. 처음에는 기본값(6,000명)으로 끝까지 한 번
돌려보고, 결론을 확인하고 싶을 때 `None`(전체 20,336명)으로 바꾸세요.

In [ ]:
import sys, warnings, time
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# ============ 설정 (서버 경로에 맞게 수정) ============
DATA_DIR  = str(REPO / "data" / "challenge2019" / "training_setA")   # p*.psv 폴더
MAX_FILES = 6000       # 읽을 환자 수. None = 전체 20,336명 (로딩 ~3분)
HORIZON   = 6          # 예측 지평(시간)
WINDOW    = 8          # 관찰 윈도우 길이(시간)
SEED      = 42         # 환자 분할 seed — 7절에서 이 값을 바꿔가며 검증합니다
USE_GPU   = False      # GPU 있으면 True
# ====================================================

VITALS = ["pulse", "sbp", "dbp", "temperature", "spo2", "resp_rate"]
print("repo:", REPO)
print("data:", DATA_DIR)

In [ ]:
# 데이터가 있는지 먼저 확인 — 없으면 받는 법을 안내하고 멈춘다
n_psv = len(list(Path(DATA_DIR).glob("*.psv"))) if Path(DATA_DIR).is_dir() else 0

if n_psv == 0:
    print("=" * 74)
    print(" Challenge-2019 데이터를 찾지 못했습니다.")
    print(f"   찾은 경로: {DATA_DIR}")
    print()
    print(" 받으세요 (공개 데이터, 로그인 불필요, ODC-BY):")
    print()
    print("   scripts/fetch_data.sh challenge2019                # 전체 20,336명, ~10분, 162MB")
    print("   scripts/fetch_data.sh challenge2019 --limit=3000   # 노트북 학습용, ~2분, 25MB")
    print()
    print(" 중간에 끊겨도 다시 실행하면 빠진 것만 이어받습니다.")
    print(" 받은 뒤 이 셀부터 다시 실행하세요.")
    print("=" * 74)
else:
    print(f"환자 파일 {n_psv:,}개 발견 → {DATA_DIR}")
    if MAX_FILES and MAX_FILES < n_psv:
        print(f"이번 실행에서는 그중 {MAX_FILES:,}명만 읽습니다 (MAX_FILES).")
    elif MAX_FILES:
        print(f"MAX_FILES={MAX_FILES:,}이지만 파일이 {n_psv:,}개뿐이라 전부 읽습니다.")
    else:
        print("전체를 읽습니다 (MAX_FILES=None). 로딩에 몇 분 걸립니다.")

## 1. 데이터 로드

### 원본이 어떻게 생겼나

Challenge 2019는 환자 한 명이 파일 하나(`p000001.psv`)이고, `|`로 구분된 표입니다.
**한 행이 한 시간**이라 04번의 긴 형태보다 다루기 쉽습니다.

```
HR|O2Sat|Temp|SBP|MAP|DBP|Resp|...|Age|Gender|ICULOS|SepsisLabel
NaN|NaN|NaN|NaN|NaN|NaN|NaN|...|53|0|1|0        ← 입실 1시간째, 아직 음성
97|95|NaN|98|75|58|19|...|53|0|2|0              ← 2시간째
102|94|37.2|95|72|55|22|...|53|0|8|1            ← 8시간째부터 SepsisLabel=1
```

핵심은 마지막 열 **`SepsisLabel`** 입니다. **시간 단위로** 0/1이 기록돼 있어서,
`cohort_from_challenge2019()`가 1이 처음 나타나는 시각을 발병 시각으로 잡습니다.
04번 데이터가 일 단위였던 것과 대비됩니다 — 이 차이가 결과를 갈랐습니다.

In [ ]:
from vitals_data import cohort_from_challenge2019

t0 = time.time()
cohort = cohort_from_challenge2019(DATA_DIR, max_files=MAX_FILES)
print(f"로딩 {time.time() - t0:.0f}초")

n_patients = cohort.vitals["patient_id"].nunique()
n_event = int(cohort.events["arrest_hour"].notna().sum())
print(f"\n환자 수        : {n_patients:,}")
print(f"발병 환자      : {n_event:,}  ({n_event / n_patients:.1%})")
print(f"활력징후 행 수 : {len(cohort.vitals):,}")
print(f"환자당 평균    : {len(cohort.vitals) / n_patients:.1f}행 (= 시간)")

In [ ]:
print("① vitals — 환자 × 시간별 활력징후")
display(cohort.vitals.head())

print("\n② events — 환자별 발병 시각 (NaN = 발병 없음 = 대조군)")
display(cohort.events.head())

print("\n③ demographics — 정적 정보")
display(cohort.demographics.head() if cohort.demographics is not None else "없음")

## 2. EDA — 발병 전에 신호가 실제로 있는가

모델을 돌리기 전에 확인할 게 있습니다. **발병 전에 활력징후가 정말 변하는가?**
안 변한다면 아무리 좋은 모델도 소용없습니다.

먼저 결측부터 봅니다. 어떤 항목이 얼마나 자주 측정되는지는 그 자체로 정보입니다.

In [ ]:
miss = cohort.vitals[VITALS].isna().mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.barh(miss.index, (1 - miss) * 100, color="steelblue")
ax.set_xlabel("measured (%)"); ax.set_xlim(0, 100)
ax.set_title("Vital-sign coverage in raw hourly rows")
for b, v in zip(bars, (1 - miss) * 100):
    ax.text(v + 1, b.get_y() + b.get_height() / 2, f"{v:.0f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()

print("측정된 비율 (원본 시간 행 기준):")
for k, v in (1 - miss).items():
    print(f"  {k:12s} {v:.1%}")

체온이 눈에 띄게 낮습니다 — 맥박은 매시간 재지만 체온은 몇 시간에 한 번 재기 때문입니다.
**이건 나중에 중요해집니다.** NEWS 점수는 체온을 쓰는데, 결측이 많으면 NEWS가 불리해질
수 있거든요. 6절에서 이게 문제인지 아닌지 직접 확인합니다.

In [ ]:
# 발병 시각을 0으로 맞춰 정렬한 뒤, 발병 전 24시간의 평균 궤적을 그린다
ev = cohort.events.dropna(subset=["arrest_hour"]).set_index("patient_id")["arrest_hour"]
v = cohort.vitals[cohort.vitals["patient_id"].isin(ev.index)].copy()
v["t_before"] = v["patient_id"].map(ev) - v["hour"]          # 발병까지 남은 시간
case = v[(v["t_before"] >= 0) & (v["t_before"] <= 24)]

ctrl_ids = cohort.events[cohort.events["arrest_hour"].isna()]["patient_id"]
ctrl_mean = cohort.vitals[cohort.vitals["patient_id"].isin(ctrl_ids)][VITALS].mean()

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, vit in zip(axes.ravel(), VITALS):
    prof = case.groupby("t_before")[vit].mean().sort_index(ascending=False)
    ax.plot(prof.index, prof.values, color="indianred", lw=2, label="sepsis cases")
    ax.axhline(ctrl_mean[vit], color="steelblue", ls="--", lw=1.5, label="controls (mean)")
    ax.invert_xaxis()                       # 왼쪽이 과거, 오른쪽(0)이 발병 시점
    ax.set_title(vit); ax.set_xlabel("hours before onset")
axes.ravel()[0].legend(fontsize=8)
plt.suptitle("Mean vital trajectory approaching sepsis onset", y=1.01)
plt.tight_layout(); plt.show()

빨간 선이 오른쪽(발병 시점)으로 갈수록 파란 점선(대조군 평균)에서 멀어지면 신호가 있는 겁니다.
맥박과 호흡수가 올라가고 혈압이 떨어지는 패턴이 보이면 정상입니다.

다만 **이건 수백 명을 평균낸 그림**이라는 걸 기억하세요. 개별 환자는 훨씬 지저분하고,
그래서 이 문제가 어려운 겁니다.

## 3. 슬라이딩 윈도우 (04번 복습)

시계열을 학습 가능한 형태로 자릅니다.

```
환자 A의 시간축 ──────────────────────────────▶  발병!
              [--8시간 관찰--]                      시각 T
                    ↑ 이 8시간을 요약해 1개 샘플로
                      라벨: "지금부터 6시간 안에 발병하는가?"
```

각 윈도우에서 vital마다 평균·표준편차·최소·최대·최근값·**기울기**·변화량을 뽑습니다.
기울기가 중요합니다 — "지금 값"보다 "어느 방향으로 얼마나 빨리 변하는가"가 악화 신호입니다.

In [ ]:
from vitals_data import build_windows, add_personalized_features

t0 = time.time()
windowed = build_windows(cohort, observation_window_hours=WINDOW, prediction_horizon_hours=HORIZON)
feat = add_personalized_features(windowed, cohort)     # 개인 기저선 대비 편차 + 나이/성별
print(f"윈도우 생성 {time.time() - t0:.0f}초\n")

BASE = float(feat.labels.mean())     # AUPRC 기준선 = 양성 비율
print(f"윈도우 개수 : {len(feat.labels):,}")
print(f"양성 윈도우 : {int(feat.labels.sum()):,}  ({BASE:.2%})")
print(f"피처 개수   : {feat.features.shape[1]}")
print(f"\n★ AUPRC 기준선 = {BASE:.4f}")
print("  이 값을 기억하세요. AUPRC는 반드시 이 기준선과 비교해서 읽습니다.")

### AUPRC를 읽는 법 — 이 노트북에서 가장 중요한 규칙

양성이 약 1%뿐입니다. 이런 데이터에서 **AUPRC 0.03은 낮은 게 아닙니다.**
아무 정보 없이 찍으면 기준선(≈0.012)이 나오니까요. 0.03이면 기준선의 2.5배입니다.

> **절대값으로 판단하지 마세요. 항상 `AUPRC ÷ 기준선`으로 보세요.**

## 4. 환자 단위 분할 (04번 복습)

한 환자에서 윈도우가 수십 개 나옵니다. 무작위로 나누면 같은 환자의 hour 10 윈도우가
train에, hour 11 윈도우가 test에 들어갑니다. **거의 같은 데이터라 모델이 답을 이미 본 셈**이고
성능이 크게 부풀려집니다. 그래서 환자를 통째로 나눕니다.

In [ ]:
from vitals_data import patient_level_split

split = patient_level_split(feat, seed=SEED)

print(f"train 윈도우 : {len(split.y_train):,}  (양성 {int(split.y_train.sum()):,})")
print(f"test  윈도우 : {len(split.y_test):,}  (양성 {int(split.y_test.sum()):,})")
print(f"\ntest 양성 비율 = {split.y_test.mean():.4f}  ← 이번 분할의 AUPRC 기준선")

BASE = float(split.y_test.mean())

## 5. XGBoost vs NEWS — 이기는가

**NEWS**(National Early Warning Score)는 실제 병동에서 쓰는 규칙 기반 점수표입니다.
맥박이 빠르면 +2점, 혈압이 낮으면 +3점 하는 식으로 각 vital을 0~3점 매겨 합산합니다.
**학습을 하지 않습니다.** 그냥 정해진 표입니다.

우리 모델은 이걸 이겨야 의미가 있습니다. 비슷하다면 복잡한 걸 새로 도입할 이유가 없으니까요.

In [ ]:
from vitals_train import train_xgboost, evaluate_news_baseline, compute_news_scores

t0 = time.time()
model, xgb_m = train_xgboost(split, use_gpu=USE_GPU, random_state=SEED)
news_m = evaluate_news_baseline(split)
print(f"학습 {time.time() - t0:.0f}초\n")

xgb_s = model.predict_proba(split.X_test)[:, 1]      # 모델 확률
news_s = compute_news_scores(split.X_test)           # NEWS 점수

res = pd.DataFrame([{
    "model": m.model_name,
    "AUPRC": round(m.auprc, 4),
    "vs baseline": f"{m.auprc / BASE:.1f}x",
    "ROC-AUC": round(m.roc_auc, 3),
    "sens@95spec": round(m.sensitivity_at_95_specificity, 3),
} for m in (xgb_m, news_m)])

print(f"AUPRC 기준선 = {BASE:.4f}\n")
display(res)

`sens@95spec`을 눈여겨보세요. **"헛알람을 아주 강하게 억제한 상태에서 위험 환자를 몇 %나
잡아내는가"** 입니다. 실제 병원이 쓰고 싶어하는 조건이 이쪽입니다.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))

for name, s, c in [("XGBoost", xgb_s, "C0"), ("NEWS", news_s, "C1")]:
    p, r, _ = precision_recall_curve(split.y_test, s)
    ax[0].plot(r, p, label=name, color=c, lw=2)
    fpr, tpr, _ = roc_curve(split.y_test, s)
    ax[1].plot(fpr, tpr, label=name, color=c, lw=2)

ax[0].axhline(BASE, color="gray", ls="--", lw=1.2, label=f"baseline {BASE:.4f}")
ax[0].set_xlabel("recall (sensitivity)"); ax[0].set_ylabel("precision")
ax[0].set_title("Precision-Recall (the honest curve for rare events)"); ax[0].legend()

ax[1].plot([0, 1], [0, 1], color="gray", ls="--", lw=1.2)
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate")
ax[1].set_title("ROC"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. 알람 부담 — 핵심 주장, 그리고 함정

### 알람 부담이란

> **두 방법의 검출률(민감도)을 똑같이 맞춘 뒤, 알람을 몇 번 울리는지 비교** — 낮을수록 좋음

간호사가 체감하는 건 AUPRC가 아니라 "오늘 알람이 몇 번 울렸나"입니다.
이게 이 프로젝트의 핵심 주장이 사는 곳입니다.

아래 셀은 검출률을 50% / 70% / 90%로 맞춰가며 비교합니다. **90%만 보지 말고
세 줄을 같이 보세요.** 여기서 함정이 드러납니다.

In [ ]:
from vitals_train import alarm_burden

rows = []
for target in (0.50, 0.70, 0.90):
    xb = alarm_burden(split.y_test, xgb_s, target_sensitivity=target)
    nb = alarm_burden(split.y_test, news_s, target_sensitivity=target)
    rows.append({
        "검출률": f"{target:.0%}",
        "XGB 알람/100": round(xb["alarms_per_100_windows"], 1),
        "XGB 특이도": round(xb["specificity"], 3),
        "NEWS 알람/100": round(nb["alarms_per_100_windows"], 1),
        "NEWS 특이도": round(nb["specificity"], 3),
        "감소율": f"{1 - xb['alarms_per_100_windows'] / nb['alarms_per_100_windows']:.0%}",
    })

burden_tbl = pd.DataFrame(rows)
display(burden_tbl)

worst = burden_tbl.iloc[burden_tbl["NEWS 특이도"].idxmin()]
print(f"NEWS 특이도가 가장 낮은 지점: 검출률 {worst['검출률']}에서 {worst['NEWS 특이도']:.3f}")

if worst["NEWS 특이도"] < 0.05:
    print("\n⚠ 특이도가 0에 가깝습니다 = NEWS가 '전원에게 알람'을 울리는 상태입니다.")
    print("  그 검출률에서 NEWS를 이기는 건 산수이지 근거가 아닙니다. 비교에서 빼세요.")
else:
    print("\n이번 표본에서는 세 지점 모두 NEWS가 정상 동작해 비교가 성립합니다.")
    print("  단, 표본을 키우면 90%에서 특이도가 0으로 무너집니다 (아래 설명 참고).")

### 함정: 검출률을 높일수록 NEWS 특이도가 무너집니다

표의 **NEWS 특이도** 열을 위에서 아래로 보세요. 검출률을 올릴수록 급격히 떨어집니다.
전체 20,336명으로 돌리면 90% 행에서 **0.000**까지 내려갑니다.
(표본이 작으면 아직 0이 아닐 수 있습니다 — 위 셀의 출력이 어느 쪽인지 알려줍니다.)

특이도 0이란 **정상인 사람을 한 명도 안 거른다**, 즉 **전원에게 알람이 울린다**는 뜻입니다.

> **"모두에게 알람 울리는 상대를 이겼다"는 건 이겨도 자랑이 아닙니다.**
> 심사에서 지적당하면 방어가 어렵습니다.

왜 이런 일이 생길까요? 두 가지 가능성이 있습니다.

1. 체온 결측이 많아서 NEWS가 불리해졌다 (2절에서 본 그것)
2. NEWS 점수가 애초에 너무 거칠어서 조절이 안 된다

**둘 중 뭔지 확인해야 합니다.** 1번이라면 우리가 NEWS를 불공정하게 다룬 것이고,
2번이라면 NEWS 자체의 한계입니다. 다음 셀에서 직접 확인합니다.

In [ ]:
# 가설 1 검증: NEWS 계산에 쓰는 값들이 실제로 채워져 있는가?
news_inputs = ["resp_rate_last", "spo2_last", "temperature_last", "sbp_last", "pulse_last"]
print("NEWS가 쓰는 값의 윈도우 단위 충족률:")
for c in news_inputs:
    print(f"  {c:18s} {split.X_test[c].notna().mean():.1%}")

print(f"\nNEWS 점수 자체의 결측: {np.isnan(news_s).mean():.1%}")

# 가설 2 검증: NEWS 점수가 몇 단계로 나뉘는가?
levels = np.unique(news_s[~np.isnan(news_s)])
print(f"\nNEWS가 가질 수 있는 값의 개수: {len(levels)}단계  →  {levels.astype(int)}")
print(f"XGBoost 확률의 값 개수      : {len(np.unique(xgb_s)):,}단계")

### 결론: 2번이었습니다

NEWS가 쓰는 값들은 윈도우 단위로 **95% 이상 채워져 있습니다.** 결측 때문이 아닙니다.
(원본 시간 행에서 체온이 34%였던 게 윈도우에서는 채워지는 이유는, 8시간 윈도우 안에서
한 번이라도 쟀으면 그 값을 쓰기 때문입니다.)

진짜 이유는 **NEWS가 십수 단계짜리 거친 자**라는 것입니다 (위 출력의 "단계" 개수를 보세요).
우리 모델은 확률이라 수천~수십만 단계로 미세 조정되지만, NEWS는 정수 칸 십여 개뿐입니다.
검출률을 높이려면 기준을 한 칸씩 내려야 하는데, 칸이 성글어서 한 칸 내릴 때마다 알람이
왕창 늘어납니다. 90%에 닿으려면 결국 맨 아래 칸까지 가야 하고, 그러면 전원이 걸립니다.

> **제안서에는 90%가 아니라 50~70% 검출률의 수치를 쓰세요.**
> 거기선 NEWS도 정상적으로 작동해서 비교가 성립합니다.

아래 그림이 이 상황을 한눈에 보여줍니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

# 왼쪽: 50% 검출률에서의 직접 비교 (비교가 성립하는 지점)
b50 = burden_tbl[burden_tbl["검출률"] == "50%"].iloc[0]
vals = [b50["XGB 알람/100"], b50["NEWS 알람/100"]]
bars = ax[0].bar(["XGBoost", "NEWS"], vals, color=["C0", "C1"])
ax[0].set_ylabel("alarms per 100 windows")
ax[0].set_title("Alarm burden at matched 50% sensitivity (lower = better)")
for b, v in zip(bars, vals):
    ax[0].text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom")

# 오른쪽: 검출률을 바꿔가며 — NEWS가 100에 붙어버리는 구간이 보인다
grid = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
for name, s, c in [("XGBoost", xgb_s, "C0"), ("NEWS", news_s, "C1")]:
    y = [alarm_burden(split.y_test, s, target_sensitivity=t)["alarms_per_100_windows"] for t in grid]
    ax[1].plot([t * 100 for t in grid], y, marker="o", label=name, color=c, lw=2)
ax[1].axhline(100, color="gray", ls=":", lw=1.2)
ax[1].text(32, 101, "everything alarms (meaningless)", fontsize=8, color="gray")
ax[1].set_xlabel("target sensitivity (%)"); ax[1].set_ylabel("alarms per 100 windows")
ax[1].set_title("Where the NEWS comparison stops being fair"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. seed 재현성 — 운인가 실력인가

여기까지는 **분할 방식 하나(SEED=42)** 로 얻은 결과입니다.
환자를 어떻게 나누느냐에 따라 결과가 흔들릴 수 있습니다.

> **한 번 잘 나온 걸 실력이라고 우기면 안 됩니다.**

실제로 04번 데이터에서는 seed만 바꿨더니 `sens@95spec`이 0.000 ↔ 0.121로 출렁였습니다.
그래서 나누는 방식을 바꿔가며 여러 번 돌려봐야 합니다.

아래 셀은 seed 3개로 반복합니다 (몇 분 걸립니다). **3번 다 이기면 실력**입니다.

In [ ]:
SEEDS = [42, 7, 123]        # 더 확실히 하려면 [42, 7, 123, 2024, 99]

rows = []
for s in SEEDS:
    t0 = time.time()
    sp = patient_level_split(feat, seed=s)
    m, xm = train_xgboost(sp, use_gpu=USE_GPU, random_state=s)
    nm = evaluate_news_baseline(sp)
    xs = m.predict_proba(sp.X_test)[:, 1]
    ns = compute_news_scores(sp.X_test)
    b = float(sp.y_test.mean())
    xb50 = alarm_burden(sp.y_test, xs, 0.50)["alarms_per_100_windows"]
    nb50 = alarm_burden(sp.y_test, ns, 0.50)["alarms_per_100_windows"]
    rows.append({
        "seed": s,
        "XGB AUPRC": round(xm.auprc, 4), "XGB 배수": round(xm.auprc / b, 1),
        "NEWS AUPRC": round(nm.auprc, 4), "NEWS 배수": round(nm.auprc / b, 1),
        "XGB sens@95spec": round(xm.sensitivity_at_95_specificity, 3),
        "NEWS sens@95spec": round(nm.sensitivity_at_95_specificity, 3),
        "알람/100 @50% XGB": round(xb50, 1), "NEWS": round(nb50, 1),
    })
    print(f"seed {s:4d} 완료 ({time.time() - t0:.0f}초)")

seed_tbl = pd.DataFrame(rows)
display(seed_tbl)

wins_auprc = int((seed_tbl["XGB AUPRC"] > seed_tbl["NEWS AUPRC"]).sum())
wins_alarm = int((seed_tbl["알람/100 @50% XGB"] < seed_tbl["NEWS"]).sum())
n = len(seed_tbl)
print(f"\nAUPRC 우세      : {wins_auprc}/{n} seed")
print(f"알람 더 적음    : {wins_alarm}/{n} seed")
print(f"\nXGB AUPRC 흔들림: ±{seed_tbl['XGB AUPRC'].std():.4f}  "
      f"(평균과의 격차 {(seed_tbl['XGB AUPRC'] - seed_tbl['NEWS AUPRC']).mean():.4f})")

### 읽는 법

**흔들림(표준편차)보다 NEWS와의 격차가 훨씬 커야** 우위가 진짜입니다.
격차가 흔들림 안에 들어가면 "차이 없음"으로 읽어야 합니다.

전체 20,336명 · seed 5개 · 튜닝 50회로 돌린 결과는 다음과 같았습니다 (참고용):

| 지표 | 우리 모델 | NEWS | 승률 |
|---|---|---|---|
| AUPRC | 0.0278 ± 0.0023 | 0.0146 ± 0.0006 | **5/5** |
| sens@95spec | 0.169 ± 0.017 | 0.050 ± 0.008 | **5/5** |
| 알람/100 @50% 검출 | **23.9** | 44.9 | **5/5** |
| 알람/100 @70% 검출 | **42.8** | 70.8 | **5/5** |

`MAX_FILES`를 줄여 돌리면 이보다 약하게 나오는 게 정상입니다. 표본이 작을수록 추정이
불안정해지니까요. **몇백 명 규모에서는 일부 seed에서 NEWS에 질 수도 있습니다** —
그게 바로 "한 번의 결과를 믿으면 안 되는" 이유이고, 이 절의 요점입니다.

1,000명으로 돌렸을 때 0.027 대 0.025로 거의 동률이던 것이, 20,336명에서는 0.029 대 0.015로
벌어졌습니다. 우리 모델은 그대로인데 **NEWS 추정치가 제자리를 찾은 것**입니다.

## 8. lead-time — 개입할 시간이 있는가

> 발병 몇 시간 전에 경보했는가 — 길수록 손쓸 여유가 생김

10분 전에 알려주면 경보해도 소용없습니다. 이 지표는 대조군과 무관하게
**"발병한 환자에서 얼마나 일찍 울렸나"** 만 봅니다.

In [ ]:
from vitals_train import lead_time_summary, threshold_at_specificity

lead = lead_time_summary(split, xgb_s, threshold_at_specificity(split.y_test, xgb_s))
if lead:
    det, tot = int(lead["detected"]), int(lead["arrest_patients"])
    print(f"검출     : {det}/{tot}명  ({det / tot:.0%})")
    print(f"중앙값   : 발병 {lead['median_lead_time_h']:.1f}시간 전")
    print(f"평균     : {lead.get('mean_lead_time_h', float('nan')):.1f}시간 전")
else:
    print("이번 분할에는 발병 환자가 부족해 lead-time을 계산할 수 없습니다.")

### 여기서 조심할 것

중앙값이 20시간을 넘는다면 좋아 보이지만, **우리는 6시간 뒤를 예측하도록 가르쳤습니다.**
6시간짜리로 학습한 모델이 27시간 전부터 울린다면 두 가지 해석이 가능합니다.

1. 정말로 일찍부터 악화 징후가 있다 (좋음)
2. 급성 악화가 아니라 **원래 중증인 환자를 골라내고 있다** (주의)

**아직 구분하지 못했습니다.** 제안서에 "27시간 전 예측"이라고 쓰기 전에
이 구분을 해야 합니다. 검출률(위 `det/tot`)도 같이 보세요 — 검출된 환자가 적으면
중앙값 자체가 노이즈입니다.

## 9. SHAP — 왜 위험하다고 판단했는가

오탐 감소만큼 중요한 게 **설명 가능성**입니다. "이 환자가 위험합니다"만으로는
간호사가 움직이지 않습니다. "맥박 기울기가 가파르고 개인 기저선 대비 혈압이
떨어져서" 라고 말해줘야 합니다.

In [ ]:
try:
    import shap
    sample = split.X_test.sample(min(2000, len(split.X_test)), random_state=SEED)
    expl = shap.TreeExplainer(model)
    sv = expl.shap_values(sample)
    shap.summary_plot(sv, sample, max_display=14, show=False)
    plt.title("What drives the risk score"); plt.tight_layout(); plt.show()
except Exception as e:
    print("SHAP 생략:", e)

읽는 법: 세로축은 피처(위일수록 영향 큼), 가로축은 예측을 민 방향(오른쪽=위험).
빨강은 그 피처 값이 큰 경우입니다.

`_change` 로 끝나는 피처가 위쪽에 있으면 **개인 기저선 이탈**이 실제로 쓰이고 있다는
뜻이고, 그게 이 프로젝트의 차별점입니다.

## 10. 정리

### 오늘 확인한 것

```
Challenge 2019 (시간 단위 라벨)
   ↓  로드 · EDA          발병 전 활력징후가 실제로 변함을 눈으로 확인
   ↓  윈도우 · 분할        04번과 동일
   ↓  XGBoost vs NEWS     이김
   ↓  알람 부담           ★ 핵심 주장 — 단, 90%에서는 비교가 성립하지 않음을 발견
   ↓  seed 재현성         ★ 한 번이 아니라 반복해서 이기는지 확인
   ↓  lead-time           길지만 해석에 주의
   ↓  SHAP                왜 위험한지 설명
```

### 04번에 없던, 여기서 새로 배운 것

| 개념 | 요점 |
|---|---|
| **라벨 해상도** | 사건 시각이 일 단위면 6시간 예측은 가르칠 수 없다. 2012이 실패한 진짜 이유 |
| **비교의 함정** | 상대가 "전원 알람" 상태면 이겨도 근거가 아니다. 특이도를 항상 같이 볼 것 |
| **seed 재현성** | 한 번의 승리는 운일 수 있다. 흔들림보다 격차가 커야 실력 |
| **거친 점수의 한계** | NEWS는 15단계라 미세 조정이 안 된다 — 그게 NEWS의 구조적 약점 |

### 결론으로 말할 수 있는 것

> 환자 2만 명 실데이터에서, **같은 검출률로 맞췄을 때 알람이 절반**이고,
> 이것이 **5개 seed 모두에서** 재현됐다.

### 말하면 안 되는 것

- 이 사건은 **패혈증**이지 심정지가 아닙니다. 경북대 심정지 성능으로 옮겨 말하면 안 됩니다.
- **90% 검출률의 알람 비교**는 쓰지 마세요 (NEWS가 전원 알람인 지점).
- **lead-time 27시간**은 아직 정체가 규명되지 않았습니다.

---

### 다음에 해볼 것

- `MAX_FILES = None` 로 전체 20,336명 실행 → 위 참고 표와 비교
- `HORIZON`을 3, 12로 바꿔 지표가 어떻게 변하는지
- 7절 `SEEDS`를 5개로 늘려 재현성 강화
- CLI로 튜닝까지: `python src/sepsis_explore.py <폴더> --horizon=6 --tune --trials=50 --gpu --seed=7`

### 데이터 받는 법

```bash
scripts/fetch_data.sh challenge2019
```

노트북만 따라갈 거라면 전체를 받을 필요 없습니다:

```bash
scripts/fetch_data.sh challenge2019 --limit=3000   # ~2분, 25MB
```

공개 데이터입니다 (ODC-BY, 로그인 불필요). 중간에 끊겨도 다시 실행하면 빠진 것만 이어받습니다.